In [1]:
def pq2tfmatrix(pq):
  import meshcat.transformations as tf
  import numpy as np
  """
  Convert a pq to a transformation matrix
  :param pq: [position, quaternion]
  :return: transformation matrix
  """
  p = pq[:3]
  q = pq[3:]
  q.insert(0, q.pop())
  return tf.quaternion_matrix(q) + np.array([[0, 0, 0, p[0]],
                                             [0, 0, 0, p[1]],
                                             [0, 0, 0, p[2]],
                                             [0, 0, 0, 0]])
    
def robotInit(numLinks, resource_path, displayInitJson, vis):
  import meshcat.geometry as g
  import numpy as np
  robot = vis['robot']
  partInitConfig = displayInitJson['part_init_config']
  for i in range(numLinks):
    robot[str(i)].set_transform(pq2tfmatrix(partInitConfig[i]))

  geometryPool = displayInitJson['geometry_pool']
  for i in range(len(geometryPool)):
    geometry = geometryPool[i]
    meshcatGeo = robot[str(geometry['part_id'])][str(geometry['geometry_id'])]
    meshcatGeo.set_transform(np.array(geometry['init_pm']).reshape(4, 4))
    if i % 2 == 1: 
      material = g.MeshPhongMaterial(color=0x0660FF)
    else:
      material = g.MeshPhongMaterial(color=0xD4D4D4)
    if i == 0:
      material = g.MeshPhongMaterial(color=0x755338)
    if(geometry['shape_type'] == 'box'):
      meshcatGeo.set_object(g.Box([geometry['length'], geometry['width'], geometry['height']]), material=material)
    elif(geometry['shape_type'] == 'capsule'):
      meshcatGeo.set_object(g.Cylinder(geometry['size'][0], geometry['size'][1]), material=material)
    elif(geometry['shape_type'] == 'sphere'):
      meshcatGeo.set_object(g.Sphere(geometry['radius']), material=material)
    elif(geometry['shape_type'] == 'mesh'):
      ext = geometry['resource_path'].split('.')[-1]
      if ext == 'stl':
        meshcatGeo.set_object(g.StlMeshGeometry.from_file(
          resource_path, geometry['resource_path']), material=material)
      elif ext == 'obj':
        meshcatGeo.set_object(g.ObjMeshGeometry.from_file(
          resource_path + geometry['resource_path']), material=material)
      else:
        print("Unknown mesh file type", ext)

def setRobotPq(numLinks, frame, pqs):
  robot = frame['robot']
  for i in range(numLinks):
    robot[str(i)].set_transform(pq2tfmatrix(pqs[i]))

def binarySearch(timeIndices, time):
  import math
  """
  Binary search to find the index of the closest time
  :param timeIndices: list of time indices
  :param time: target time
  :return: index of the closest time index
  """
  low = 0
  high = len(timeIndices) - 1
  while low <= high:
    mid = (low + high) // 2
    if math.isclose(timeIndices[mid], time):
      return mid
    if timeIndices[mid] < time:
      low = mid + 1
    else:
      high = mid - 1

  return min(int(low), len(timeIndices) - 1)

def animateRobotByRecords(numLinks, records, frameRate, vis):
  from meshcat.animation import Animation
  partpq = records['partPq']
  timeIndices = records['timeIndex']
  anim = Animation()
  anim.default_framerate = frameRate

  minTime = 0
  maxTime = timeIndices[-1]
  totalFrameNumber = int((maxTime - minTime) * anim.default_framerate)

  for i in range(totalFrameNumber):
    currentTime = minTime + i / anim.default_framerate
    currentIdx = binarySearch(timeIndices, currentTime)
    with anim.at_frame(vis, i) as frame:
      setRobotPq(numLinks, frame, partpq[currentIdx])

  vis.set_animation(anim)

In [2]:
import sys
sys.path.append("D:/code/sire/install/python/release")
import sire

In [3]:
from os.path import abspath
import os
cs = sire.ControlServer.instance()
print(abspath(os.getcwd()) + "/box.xml")
sire.fromXmlFile(cs, abspath(os.getcwd()) + "/box.xml")
cs.init()

d:\code\sire\demo\demo_python/box.xml


In [4]:
simulator = sire.simulator(cs)
while(not simulator.isTimeout() and not simulator.isEventListEmpty()):
  simulator.step(1, False)
  # print("Simulation time", simulator.simTime())

RuntimeError: Failure at D:\code\sire\src\physics\physics_engine.cpp:483 in cptContactInfo(): condition 'fn.size() >= num_contacts' failed.

In [ ]:
model = cs.model()
displayInitJson = model.displayInitJson()
result = simulator.recordsToJson()

In [ ]:
import meshcat
displayInitJson
vis = meshcat.Visualizer()
resourcePath = "D:/code/sire/web_interface/public"
robotInit(model.numLinks(), resourcePath, displayInitJson, vis)
animateRobotByRecords(model.numLinks(), result, 1000, vis)
vis.jupyter_cell()

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7004/static/


In [ ]:
import matplotlib.pyplot as plt
timeIndices = result['timeIndex']
contactInfo = result["contactInfo"]
partpq = result["partPq"]
partvs = result["partVs"]
partas = result["partAs"]
x = []
y = []

for i in range(1000):
  currentTime = 6 + i / 1000
  currentIdx = binarySearch(timeIndices, currentTime)
  print(currentIdx, partpq[currentIdx][1], partvs[currentIdx][1], partas[currentIdx][1], contactInfo[currentIdx])

6010 [3.063973828414619e-22, -2.335059597160776e-19, 0.4992640928504717, -2.6175131910501635e-18, -1.7311589662393915e-17, -4.7813731575224505e-35, 1.0] [1.8758486935307905e-17, -5.42661398587255e-19, 9.749702138510336e-12, -1.0070372966111455e-18, -3.757216837688583e-17, -9.815518481770624e-36] [-1.1085904816398063e-14, 5.554369864344824e-15, 3.552713678800501e-15, 1.1125113830303743e-14, 2.2204490519446654e-14, 2.0721188867218404e-33] [{'contactForce': [-0.0, 0.0, 2.4499999999982456], 'contactWrench': [-0.0, 0.0, 2.4499999999982456, 1.2249999999991228, 1.2249999999991228, 0.0], 'contact_point_pe': [-0.5, 0.5, -0.000367953574764171, 2.356194490192345, 0.0, 5.497787143782138], 'partId_A': 0, 'partId_B': 1, 'point_pair': {'depth': 1.4178185409052801e-07, 'geomIdA': 0, 'geomIdB': 3}, 'separation_speed': 0.0, 'slip_speed': 0.0}, {'contactForce': [-0.0, 0.0, 2.450000000001758], 'contactWrench': [-0.0, 0.0, 2.450000000001758, -1.225000000000879, 1.225000000000879, -0.0], 'contact_point_pe':